# Phishing Website Detection Using Machine Learning
Corrected final pipeline. `class` is the explicit target and `Index` is excluded.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

## 1. Load and inspect dataset

In [ ]:
df = pd.read_csv("../data/phishing.csv")
print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df["class"].value_counts())

## 2. Remove Index and engineer features

In [ ]:
original_features = [c for c in df.columns if c not in ["Index", "class"]]
data = df.drop(columns=["Index"]).copy()
data["Suspicion Score"] = (data[original_features] == -1).sum(axis=1)
data["Safe Score"] = (data[original_features] == 1).sum(axis=1)
data["Neutral Score"] = (data[original_features] == 0).sum(axis=1)

## 3. Define target and split

In [ ]:
X = data.drop(columns=["class"])
y = data["class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4. Baseline model results

In [ ]:
# Train Logistic Regression, Random Forest, SVM, KNN and Isolation Forest here.
# See ../results/model_comparison.csv for the captured final output.

### Captured baseline output

```
              Model  Accuracy  Precision  Recall  F1 Score  Phishing Recall
Logistic Regression    0.9403     0.9403  0.9403    0.9403           0.9265
      Random Forest    0.9738     0.9739  0.9738    0.9737           0.9602
                SVM    0.9629     0.9630  0.9629    0.9629           0.9459
                KNN    0.9448     0.9448  0.9448    0.9448           0.9346
   Isolation Forest    0.4772     0.4777  0.4772    0.4774           0.4147
```

## 5. Hyperparameter tuning

In [ ]:
param_distributions = {
    "n_estimators": [150, 250, 350],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", None],
    "bootstrap": [True, False]
}
# RandomizedSearchCV uses weighted F1 on the training data.

### Captured tuned Random Forest output

Best parameters: `{'n_estimators': 250, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20, 'bootstrap': False}`

Accuracy: **0.9747**  
F1: **0.9746**  
Phishing recall: **0.9622**

## 6. Feature selection

Top 15 features:

- HTTPS
- AnchorURL
- Safe Score
- Suspicion Score
- WebsiteTraffic
- SubDomains
- LinksInScriptTags
- PrefixSuffix-
- Neutral Score
- LinksPointingToPage
- RequestURL
- DomainRegLen
- ServerFormHandler
- AgeofDomain
- UsingIP

Top-15 F1: **0.9692**

## Final conclusion
The tuned Random Forest is the final best model. The top-15 model retains nearly the same performance using fewer inputs.